# Week 4, Notebook 2: Deep Learning with Keras

**From one neuron to a real network that predicts Caribbean World Cup matches.**

*Caribbean AI - Adrian Dunkley*

---

### What you will build
- A **deep neural network** using **Keras**, the same tool used in real AI labs.
- A model that predicts whether a Caribbean team **wins, draws, or loses**.
- The habits every deep-learning project shares: scaling, a validation split,
  reading a loss curve, and beating a baseline.

### The big idea
Last notebook you built **one neuron** by hand. A single neuron can only draw a
straight dividing line, so it misses patterns that bend. The fix is to stack
**many neurons in layers**. When you have more than one layer between the input
and the output, people call it a **deep** network, and training it is **deep
learning**.

Wiring dozens of neurons by hand would be miserable, so we let **Keras** do it.
You describe the layers; Keras handles the maths you wrote out last time.

### The data: Caribbean international football

We have a table of **1,600 past matches** between Caribbean national teams
(Jamaica, Trinidad and Tobago, Haiti, Curacao, Cuba, Guyana, Barbados, and many
more). For each match we know a few numbers before kick-off, and the result.

| Column | What it means |
| --- | --- |
| `home_rating` / `away_rating` | Team strength (higher is stronger). |
| `home_recent_form` / `away_recent_form` | Momentum lately (+ good, - poor). |
| `home_goals_avg` / `away_goals_avg` | Goals per game each side usually scores. |
| `result` | What happened: `home_win`, `draw`, or `away_win`. |

Our job: learn to predict `result` from the numbers we know before kick-off.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

matches = pd.read_csv("data/caribbean_matches.csv")
print("Matches:", len(matches))
matches.head()

In [ ]:
# How often does each result happen? This sets our baseline to beat.
counts = matches["result"].value_counts()
print(counts)
baseline = counts.max() / len(matches)
print(f"\nBaseline (always guess '{counts.idxmax()}'): {baseline:.1%}")

### Step 1: Choose the inputs (X) and the answer (y)

`X` holds the clues we feed in. `y` holds the answer we want out. The result is a
word (`home_win`, `draw`, `away_win`), but a network only speaks numbers, so we
turn each word into a number code: 0, 1, 2.

In [ ]:
feature_cols = ["home_rating", "away_rating", "home_recent_form",
                "away_recent_form", "home_goals_avg", "away_goals_avg"]
X = matches[feature_cols].values

# Map the three results to numbers the network can output.
label_names = ["home_win", "draw", "away_win"]
label_to_num = {name: i for i, name in enumerate(label_names)}
y = matches["result"].map(label_to_num).values

print("X shape:", X.shape, "  (rows = matches, columns = clues)")
print("First result as a number:", y[0], "=", label_names[y[0]])

### Step 2: Split into training and test sets

We hide some matches from the network while it learns (the **test set**), then
check it on those unseen matches. This is how we know it truly learned the game
and did not just memorise. We use scikit-learn's splitter for this.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Scale features: fit the scaler on TRAIN only, then apply to both.
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

print("Training matches:", len(X_train), " Test matches:", len(X_test))

### Step 3: Build the deep network with Keras

Here is where Keras shines. We stack layers with `Sequential`:

- **Input**: 6 numbers (our features).
- **Hidden layer 1**: 32 neurons. Each is exactly the neuron you built by hand.
- **Hidden layer 2**: 16 neurons. A second layer lets the network combine ideas
  from the first, which is what "deep" buys us.
- **Output**: 3 neurons, one per possible result, with **softmax** so they turn
  into three probabilities that add up to 100%.

`relu` is a modern squashing function for hidden layers (it simply turns
negatives into 0). It trains faster than the sigmoid for deep networks.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)   # so everyone gets the same result

model = keras.Sequential([
    layers.Input(shape=(6,)),          # 6 features come in
    layers.Dense(32, activation="relu"),   # hidden layer 1: 32 neurons
    layers.Dense(16, activation="relu"),   # hidden layer 2: 16 neurons
    layers.Dense(3, activation="softmax"), # 3 outputs: win / draw / loss
])

model.summary()

### Step 4: Compile and train

Before training we **compile** the model, telling it:

- **optimizer** `adam`: the smart version of the gradient descent you coded by
  hand. It decides how big each step should be.
- **loss** `sparse_categorical_crossentropy`: the score for "how wrong" when the
  answer is one of several classes. The network tries to make this small.
- **metrics** `accuracy`: the human-friendly score we watch.

Then `fit` runs the training loop. Each **epoch** is one full pass over the
training matches. We keep 15% aside as a **validation** set to watch for
over-fitting (memorising instead of learning).

In [ ]:
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

history = model.fit(
    X_train_s, y_train,
    validation_split=0.15,
    epochs=60,
    batch_size=32,
    verbose=0,          # quiet training; we will plot the curves instead
)
print("Training finished.")
print("Final training accuracy:  ", round(history.history["accuracy"][-1], 3))
print("Final validation accuracy:", round(history.history["val_accuracy"][-1], 3))

### Step 5: Read the learning curves

Two curves tell the whole story:

- **Loss going down** = the network is learning.
- **Training vs validation** = if training accuracy climbs but validation
  stalls or drops, the network is starting to **memorise** instead of learn
  (over-fitting). Ours should track fairly closely.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history["loss"], label="training")
ax1.plot(history.history["val_loss"], label="validation")
ax1.set_title("Loss (lower is better)")
ax1.set_xlabel("epoch"); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(history.history["accuracy"], label="training")
ax2.plot(history.history["val_accuracy"], label="validation")
ax2.set_title("Accuracy (higher is better)")
ax2.set_xlabel("epoch"); ax2.legend(); ax2.grid(alpha=0.3)
plt.show()

### Step 6: Score on the unseen test matches

Now the real exam: matches the network never saw during training. We compare its
accuracy to the baseline from the top of the notebook.

In [ ]:
test_loss, test_acc = model.evaluate(X_test_s, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.1%}")
print(f"Baseline:      {baseline:.1%}")
print("The network beat the baseline!" if test_acc > baseline
      else "No better than guessing - needs work.")

### Which mistakes does it make? A confusion matrix

Accuracy is one number, but *how* it is wrong matters. A **confusion matrix**
shows, for each true result, what the network predicted. Draws are famously the
hardest to call, in Caribbean football and everywhere else.

In [ ]:
from sklearn.metrics import confusion_matrix

pred_classes = model.predict(X_test_s, verbose=0).argmax(axis=1)
cm = confusion_matrix(y_test, pred_classes)

fig, ax = plt.subplots(figsize=(5.5, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(3)); ax.set_xticklabels(label_names, rotation=20)
ax.set_yticks(range(3)); ax.set_yticklabels(label_names)
ax.set_xlabel("network predicted"); ax.set_ylabel("actually happened")
ax.set_title("Confusion matrix")
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max()/2 else "black")
plt.show()

### Step 7: Predict real upcoming fixtures

Here is the payoff. We load fixtures that have **not been played** and ask the
network for the win/draw/loss chances. Remember to scale them with the **same**
scaler the network trained on.

In [ ]:
upcoming = pd.read_csv("data/caribbean_upcoming_matches.csv")
Xu = scaler.transform(upcoming[feature_cols].values)
probs = model.predict(Xu, verbose=0)

print("Upcoming Caribbean fixtures - the network's call:\n")
for i, row in upcoming.iterrows():
    p = probs[i]
    call = label_names[p.argmax()]
    home, away = row["home_team"], row["away_team"]
    print(f"{home} vs {away}")
    print(f"   {home} win: {p[0]:.0%}   draw: {p[1]:.0%}   {away} win: {p[2]:.0%}"
          f"   ->  most likely: {call}\n")

### What you learned

- A **deep network** is neurons stacked in **layers**; Keras builds it from a
  short description while doing the maths you coded by hand last time.
- The workflow is always the same: **split**, **scale**, **build**, **compile**,
  **fit**, then **read the curves** and **beat the baseline**.
- **softmax** turns the final layer into probabilities; a **confusion matrix**
  shows which results the model confuses.
- Football has real randomness, so even a good model lands in the 60s for
  accuracy. That is honest: no model predicts sport perfectly.

### Try it yourself
1. Add a third hidden layer, or change `32` and `16` to bigger numbers. Does the
   test accuracy improve, or does the model start to over-fit?
2. Train for `200` epochs and watch for the validation loss turning back up.
3. Invent your own fixture (pick two Caribbean teams and made-up form numbers)
   and ask the model who wins.